In [3]:
import kagglehub
import shutil
import os

path = kagglehub.dataset_download("olistbr/brazilian-ecommerce")

dest = r"C:\Projects\Olist\data"    

os.makedirs(dest, exist_ok=True)
shutil.copytree(path, dest, dirs_exist_ok=True)

print(os.listdir(dest))

Resuming download from 7340032 bytes (37377548 bytes left)...
Resuming download to C:\Users\harba\.cache\kagglehub\datasets\olistbr\brazilian-ecommerce\2.archive (7340032/44717580) bytes left.


100%|██████████| 42.6M/42.6M [00:37<00:00, 1.01MB/s]

Extracting files...


['olist_customers_dataset.csv', 'olist_geolocation_dataset.csv', 'olist_orders_dataset.csv', 'olist_order_items_dataset.csv', 'olist_order_payments_dataset.csv', 'olist_order_reviews_dataset.csv', 'olist_products_dataset.csv', 'olist_sellers_dataset.csv', 'product_category_name_translation.csv']


In [7]:
import pandas as pd
import os

data_path = r"C:\Projects\Olist\data"

files = {
    "orders": "olist_orders_dataset.csv",
    "order_items": "olist_order_items_dataset.csv",
    "order_payments": "olist_order_payments_dataset.csv",
    "order_reviews": "olist_order_reviews_dataset.csv",
    "customers": "olist_customers_dataset.csv",
    "products": "olist_products_dataset.csv",
    "sellers": "olist_sellers_dataset.csv",
    "geolocation": "olist_geolocation_dataset.csv",
    "category_translation": "product_category_name_translation.csv"
}

dfs = {}
for name, file in files.items():
    dfs[name] = pd.read_csv(os.path.join(data_path, file))
    print(f"{name}: {dfs[name].shape} — {dfs[name].isnull().sum().sum()} nulls")

orders: (99441, 8) — 4908 nulls
order_items: (112650, 7) — 0 nulls
order_payments: (103886, 5) — 0 nulls
order_reviews: (99224, 7) — 145903 nulls
customers: (99441, 5) — 0 nulls
products: (32951, 9) — 2448 nulls
sellers: (3095, 4) — 0 nulls
geolocation: (1000163, 5) — 0 nulls
category_translation: (71, 2) — 0 nulls


In [6]:
import os

data_path = r"C:\Projects\Olist\data"

for f in os.listdir(data_path):
    print(f)

olist_customers_dataset.csv
olist_geolocation_dataset.csv
olist_orders_dataset.csv
olist_order_items_dataset.csv
olist_order_payments_dataset.csv
olist_order_reviews_dataset.csv
olist_products_dataset.csv
olist_sellers_dataset.csv
product_category_name_translation.csv


In [8]:
# Orders — which columns have nulls?
print("=== ORDERS ===")
print(dfs["orders"].isnull().sum())
print("\n=== PRODUCTS ===")
print(dfs["products"].isnull().sum())
print("\n=== REVIEWS ===")
print(dfs["order_reviews"].isnull().sum())
print("\n=== ORDERS dtypes ===")
print(dfs["orders"].dtypes)

=== ORDERS ===
order_id                            0
customer_id                         0
order_status                        0
order_purchase_timestamp            0
order_approved_at                 160
order_delivered_carrier_date     1783
order_delivered_customer_date    2965
order_estimated_delivery_date       0
dtype: int64

=== PRODUCTS ===
product_id                      0
product_category_name         610
product_name_lenght           610
product_description_lenght    610
product_photos_qty            610
product_weight_g                2
product_length_cm               2
product_height_cm               2
product_width_cm                2
dtype: int64

=== REVIEWS ===
review_id                      0
order_id                       0
review_score                   0
review_comment_title       87656
review_comment_message     58247
review_creation_date           0
review_answer_timestamp        0
dtype: int64

=== ORDERS dtypes ===
order_id                         str
customer_i

In [14]:
# Clean Orders
orders_clean = dfs["orders"].copy()

# Convert datetime columns to date formatting
date_cols = ['order_purchase_timestamp', 'order_approved_at', 'order_delivered_carrier_date', 'order_delivered_customer_date', 'order_estimated_delivery_date']
for col in date_cols:
    orders_clean[col] = pd.to_datetime(orders_clean[col], errors='coerce')

# Drop rows where order_id or customer_id is null (useless rows without these identifiers)
orders_clean = orders_clean.dropna(subset=['order_id', 'customer_id'])

print("Orders cleaned:", orders_clean.shape)
print("Order statuses:", orders_clean['order_status'].value_counts())

# Clean Products
products_clean = dfs["products"].copy()

# Drop duplicates by product_id, keep first
products_clean = products_clean.drop_duplicates(subset=['product_id'], keep='first')

# Fill missing category names with 'unknown'
products_clean['product_category_name'] = products_clean['product_category_name'].fillna('unknown')

# Fill dimension/weight nulls with 0
dimension_cols = ['product_length_cm', 'product_height_cm', 'product_width_cm', 'product_weight_g']
for col in dimension_cols:
    products_clean[col] = products_clean[col].fillna(0)

# Ensure positive dimensions for consistency (negative values are likely data entry errors)
for col in dimension_cols:
    products_clean = products_clean[products_clean[col] >= 0]

print("\nProducts cleaned:", products_clean.shape)

# Clean Reviews
reviews_clean = dfs["order_reviews"].copy()

# Convert datetime columns to date formatting
reviews_clean['review_creation_date'] = pd.to_datetime(reviews_clean['review_creation_date'], errors='coerce')
reviews_clean['review_answer_timestamp'] = pd.to_datetime(reviews_clean['review_answer_timestamp'], errors='coerce')

# Fill missing comments with empty string
reviews_clean['review_comment_title'] = reviews_clean['review_comment_title'].fillna('')
reviews_clean['review_comment_message'] = reviews_clean['review_comment_message'].fillna('')

# Validate review scores (1-5)
reviews_clean = reviews_clean[(reviews_clean['review_score'] >= 1) & (reviews_clean['review_score'] <= 5)]

print("\nReviews cleaned:", reviews_clean.shape)
print("Review scores:", reviews_clean['review_score'].value_counts().sort_index())

# Clean Order Items
order_items_clean = dfs["order_items"].copy()

# Drop nulls
order_items_clean = order_items_clean.dropna()

# Validate positive prices and freight
order_items_clean = order_items_clean[(order_items_clean['price'] > 0) & (order_items_clean['freight_value'] >= 0)]

print("\nOrder Items cleaned:", order_items_clean.shape)

# Clean Payments
payments_clean = dfs["order_payments"].copy()

# Drop nulls
payments_clean = payments_clean.dropna()

# Validate positive payment values
payments_clean = payments_clean[payments_clean['payment_value'] > 0]

# Ensure non-negative installments
payments_clean = payments_clean[payments_clean['payment_installments'] >= 0]

print("\nPayments cleaned:", payments_clean.shape)
print("Payment types:", payments_clean['payment_type'].value_counts())

# Clean Customers
customers_clean = dfs["customers"].copy()

# Remove duplicates by customer_id
customers_clean = customers_clean.drop_duplicates(subset=['customer_id'], keep='first')

# Standardize strings
customers_clean['customer_city'] = customers_clean['customer_city'].str.lower().str.strip()
customers_clean['customer_state'] = customers_clean['customer_state'].str.upper().str.strip()

print("\nCustomers cleaned:", customers_clean.shape)

# Clean Sellers
sellers_clean = dfs["sellers"].copy()

# Remove duplicates by seller_id
sellers_clean = sellers_clean.drop_duplicates(subset=['seller_id'], keep='first')

# Standardize strings
sellers_clean['seller_city'] = sellers_clean['seller_city'].str.lower().str.strip()
sellers_clean['seller_state'] = sellers_clean['seller_state'].str.upper().str.strip()

print("\nSellers cleaned:", sellers_clean.shape)

# Clean Geolocation
geolocation_clean = dfs["geolocation"].copy()

# Remove duplicates by zip code prefix
geolocation_clean = geolocation_clean.drop_duplicates(subset=['geolocation_zip_code_prefix'], keep='first')

# Drop rows with missing coordinates
geolocation_clean = geolocation_clean.dropna(subset=['geolocation_lat', 'geolocation_lng'])

# Standardize strings
geolocation_clean['geolocation_state'] = geolocation_clean['geolocation_state'].str.upper().str.strip()
geolocation_clean['geolocation_city'] = geolocation_clean['geolocation_city'].str.lower().str.strip()

print("\nGeolocation cleaned:", geolocation_clean.shape)

# Clean Category Translation
category_clean = dfs["category_translation"].copy()

# Remove duplicates
category_clean = category_clean.drop_duplicates()

# Strip whitespace
category_clean['product_category_name'] = category_clean['product_category_name'].str.strip()
category_clean['product_category_name_english'] = category_clean['product_category_name_english'].str.strip()

print("\nCategory Translation cleaned:", category_clean.shape)

# Summary
print("\n" + "="*60)
print("CLEANING SUMMARY")
print("="*60)
print(f"Orders: {dfs['orders'].shape[0]} → {orders_clean.shape[0]}")
print(f"Products: {dfs['products'].shape[0]} → {products_clean.shape[0]}")
print(f"Reviews: {dfs['order_reviews'].shape[0]} → {reviews_clean.shape[0]}")
print(f"Order Items: {dfs['order_items'].shape[0]} → {order_items_clean.shape[0]}")
print(f"Payments: {dfs['order_payments'].shape[0]} → {payments_clean.shape[0]}")
print(f"Customers: {dfs['customers'].shape[0]} → {customers_clean.shape[0]}")
print(f"Sellers: {dfs['sellers'].shape[0]} → {sellers_clean.shape[0]}")
print(f"Geolocation: {dfs['geolocation'].shape[0]} → {geolocation_clean.shape[0]}")

Orders cleaned: (99441, 8)
Order statuses: order_status
delivered      96478
shipped         1107
canceled         625
unavailable      609
invoiced         314
processing       301
created            5
approved           2
Name: count, dtype: int64

Products cleaned: (32951, 9)

Reviews cleaned: (99224, 7)
Review scores: review_score
1    11424
2     3151
3     8179
4    19142
5    57328
Name: count, dtype: int64

Order Items cleaned: (112650, 7)

Payments cleaned: (103877, 5)
Payment types: payment_type
credit_card    76795
boleto         19784
voucher         5769
debit_card      1529
Name: count, dtype: int64

Customers cleaned: (99441, 5)

Sellers cleaned: (3095, 4)

Geolocation cleaned: (19015, 5)

Category Translation cleaned: (71, 2)

CLEANING SUMMARY
Orders: 99441 → 99441
Products: 32951 → 32951
Reviews: 99224 → 99224
Order Items: 112650 → 112650
Payments: 103886 → 103877
Customers: 99441 → 99441
Sellers: 3095 → 3095
Geolocation: 1000163 → 19015


In [15]:
# Save cleaned datasets to CSV
output_path = r"C:\Projects\Olist\data\cleaned"
os.makedirs(output_path, exist_ok=True)

orders_clean.to_csv(os.path.join(output_path, "orders_clean.csv"), index=False)
products_clean.to_csv(os.path.join(output_path, "products_clean.csv"), index=False)
reviews_clean.to_csv(os.path.join(output_path, "reviews_clean.csv"), index=False)
order_items_clean.to_csv(os.path.join(output_path, "order_items_clean.csv"), index=False)
payments_clean.to_csv(os.path.join(output_path, "payments_clean.csv"), index=False)
customers_clean.to_csv(os.path.join(output_path, "customers_clean.csv"), index=False)
sellers_clean.to_csv(os.path.join(output_path, "sellers_clean.csv"), index=False)
geolocation_clean.to_csv(os.path.join(output_path, "geolocation_clean.csv"), index=False)
category_clean.to_csv(os.path.join(output_path, "category_clean.csv"), index=False)

print(f"\n✓ All cleaned data saved to: {output_path}")


✓ All cleaned data saved to: C:\Projects\Olist\data\cleaned
